# BigQuery 속성 그래프 구축, GQL 분석 및 네이티브 시각화 실습

이 노트북은 BigQuery Property Graph의 구축부터 시작하여, GQL(Graph Query Language)을 이용한 다중 홉(Multi-Hop) 관계 분석 및 Colab Enterprise의 네이티브 그래프 시각화 기능(`--graph display_only`)을 활용한 시각화까지의 과정을 핵심만 간결하게 다룹니다.

특정 인기 상품(Product)을 중심으로 이를 구매한 여러 고객(Customer), 고객들의 거주 지역(Location), 그리고 상품이 공급되는 물류센터(Distribution Center) 간의 유기적인 네트워크 관계를 한눈에 시각화합니다.

### 학습 목표
1. **환경 초기화 및 라이브러리 로드**: BigQuery 클라이언트를 초기화하고 REST API 사용을 위한 인증 및 환경설정을 진행합니다.
2. **속성 그래프(Property Graph)용 노드 및 엣지 테이블 구축**: 정제된 데이터를 바탕으로 그래프 모델링을 위한 노드(Node) 및 엣지(Edge) 테이블을 구축하는 방법을 이해합니다.
3. **BigQuery Property Graph 생성**: 구축된 테이블들 간의 관계를 바탕으로 BigQuery 내에 Property Graph를 생성하는 방법을 학습합니다.
4. **GQL 분석 및 네이티브 시각화**: GQL(Graph Query Language)을 사용하여 다중 홉 관계를 분석하고, Colab Enterprise의 네이티브 그래프 시각화 기능을 활용하는 방법을 실습합니다.

### 작업 파이프라인
1. **Step 1: 초기 환경 설정**: BigQuery 클라이언트 사용을 위한 라이브러리 로드 및 인증 설정을 확인합니다.
2. **Step 2: 속성 그래프(Property Graph)용 노드 및 엣지 테이블 구축**: 상품, 고객, 주소(Location), 물류센터(Distribution Center) 테이블과 이를 연결하는 주문(Order), 공급(Supply) 관계 테이블을 구축합니다.
3. **Step 3: BigQuery Property Graph 생성**: 생성된 노드 및 엣지 테이블들의 관계를 선언하는 Property Graph를 생성합니다.
4. **Step 4: GQL 분석 및 네이티브 시각화**: GQL 쿼리를 통해 특정 상품을 중심으로 한 네트워크를 분석하고 시각화합니다.

## Step 1: 초기 환경 설정

BigQuery 클라이언트 라이브러리를 임포트하고 인증 정보를 로드하여 실습을 위한 초기 환경을 설정합니다.

In [ ]:
# 필요 라이브러리 및 확장 로드
%load_ext google.cloud.bigquery

import google.auth
from google.auth.transport.requests import Request
from google.cloud import bigquery

# 기본 설정
LOCATION = "us-central1"
credentials, PROJECT_ID = google.auth.default()
credentials.refresh(Request())

print(f"GCP Project ID: {PROJECT_ID}")

## Step 2: 속성 그래프(Property Graph)용 노드 및 엣지 테이블 구축

정제 레이어용 데이터셋(`thelook_network`)을 생성하고, 그래프를 구성할 노드(Customer, Product, Location, DC) 및 엣지(ordered, co_purchased, lives_in, supplied_from) 테이블을 생성합니다.

In [ ]:
from google.api_core.exceptions import Conflict

client = bigquery.Client()

# 0. 정제 레이어용 데이터셋 생성 (존재하지 않을 경우)
dataset_id = f"{PROJECT_ID}.thelook_network"
dataset = bigquery.Dataset(dataset_id)
dataset.location = LOCATION  # us-central1

try:
    client.create_dataset(dataset, timeout=30)
    print(f"데이터셋 '{dataset_id}'이 생성되었습니다.")
except Conflict:
    print(f"데이터셋 '{dataset_id}'이 이미 존재합니다.")

# 1. 노드 및 엣지 테이블 생성 SQL
setup_tables_sql = f"""
-- 1) 노드 테이블 생성
CREATE OR REPLACE TABLE `{PROJECT_ID}.thelook_network.node_users` AS
SELECT id AS user_id, age, gender, traffic_source
FROM `{PROJECT_ID}.thelook_ecommerce.users`;

CREATE OR REPLACE TABLE `{PROJECT_ID}.thelook_network.node_products` AS
SELECT id AS product_id, category, brand, name AS product_name, retail_price
FROM `{PROJECT_ID}.thelook_ecommerce.products`;

CREATE OR REPLACE TABLE `{PROJECT_ID}.thelook_network.node_locations` AS
SELECT 
  ROW_NUMBER() OVER (ORDER BY country, city) AS location_id,
  city, country
FROM (
  SELECT DISTINCT city, country 
  FROM `{PROJECT_ID}.thelook_ecommerce.users`
  WHERE city IS NOT NULL AND country IS NOT NULL
);

CREATE OR REPLACE TABLE `{PROJECT_ID}.thelook_network.node_distribution_centers` AS
SELECT id AS center_id, name AS center_name
FROM `{PROJECT_ID}.thelook_ecommerce.distribution_centers`;

-- 2) 엣지 테이블 생성 (각 테이블별 고유 ID 컬럼 필수 포함)
CREATE OR REPLACE TABLE `{PROJECT_ID}.thelook_network.edge_ordered` AS
SELECT 
  id AS order_item_id, user_id, product_id, sale_price, status
FROM `{PROJECT_ID}.thelook_ecommerce.order_items`;

CREATE OR REPLACE TABLE `{PROJECT_ID}.thelook_network.edge_lives_in` AS
SELECT 
  CONCAT(u.id, '_lives_in_', l.location_id) AS edge_id,
  u.id AS user_id,
  l.location_id
FROM `{PROJECT_ID}.thelook_ecommerce.users` u
JOIN `{PROJECT_ID}.thelook_network.node_locations` l
  ON u.city = l.city AND u.country = l.country;

CREATE OR REPLACE TABLE `{PROJECT_ID}.thelook_network.edge_supplied_from` AS
SELECT 
  CONCAT(id, '_supplied_from_', distribution_center_id) AS edge_id,
  id AS product_id,
  distribution_center_id AS center_id
FROM `{PROJECT_ID}.thelook_ecommerce.products`;

-- 동시구매 엣지 생성
CREATE OR REPLACE TABLE `{PROJECT_ID}.thelook_network.edge_co_purchased` AS
WITH product_pairs AS (
  SELECT 
    a.product_id AS source_product_id,
    b.product_id AS target_product_id,
    COUNT(*) AS co_purchase_count
  FROM `{PROJECT_ID}.thelook_ecommerce.order_items` a
  JOIN `{PROJECT_ID}.thelook_ecommerce.order_items` b 
    ON a.order_id = b.order_id AND a.product_id < b.product_id
  GROUP BY 1, 2
)
SELECT 
  CONCAT(source_product_id, '_', target_product_id, '_copurchase') AS edge_id,
  source_product_id,
  target_product_id,
  co_purchase_count
FROM product_pairs
WHERE co_purchase_count >= 2;
"""

print("노드 및 엣지 테이블을 생성 중입니다...")
client.query(setup_tables_sql).result()
print("모든 노드 및 엣지 테이블이 성공적으로 생성되었습니다.")

## Step 3: BigQuery Property Graph 생성

생성된 노드 및 엣지 테이블들의 관계(Source/Destination Key 매핑)를 정의하여 속성 그래프 `product_recommendation_graph`를 등록합니다.

In [ ]:
create_graph_sql = f"""
CREATE OR REPLACE PROPERTY GRAPH `{PROJECT_ID}.thelook_network.product_recommendation_graph`
NODE TABLES (
  `{PROJECT_ID}.thelook_network.node_products` AS Product KEY (product_id)
    PROPERTIES (product_id, category, brand, product_name, retail_price),
  `{PROJECT_ID}.thelook_network.node_users` AS Customer KEY (user_id)
    PROPERTIES (user_id, age, gender, traffic_source),
  `{PROJECT_ID}.thelook_network.node_locations` AS Location KEY (location_id)
    PROPERTIES (city, country),
  `{PROJECT_ID}.thelook_network.node_distribution_centers` AS DistributionCenter KEY (center_id)
    PROPERTIES (center_id, center_name)
)
EDGE TABLES (
  `{PROJECT_ID}.thelook_network.edge_ordered` AS ordered
    KEY (order_item_id)
    SOURCE KEY (user_id) REFERENCES Customer (user_id)
    DESTINATION KEY (product_id) REFERENCES Product (product_id)
    PROPERTIES (order_item_id, sale_price, status),
  `{PROJECT_ID}.thelook_network.edge_co_purchased` AS co_purchased_with
    KEY (edge_id)
    SOURCE KEY (source_product_id) REFERENCES Product (product_id)
    DESTINATION KEY (target_product_id) REFERENCES Product (product_id)
    PROPERTIES (co_purchase_count),
  `{PROJECT_ID}.thelook_network.edge_lives_in` AS lives_in
    KEY (edge_id)
    SOURCE KEY (user_id) REFERENCES Customer (user_id)
    DESTINATION KEY (location_id) REFERENCES Location (location_id),
  `{PROJECT_ID}.thelook_network.edge_supplied_from` AS supplied_from
    KEY (edge_id)
    SOURCE KEY (product_id) REFERENCES Product (product_id)
    DESTINATION KEY (center_id) REFERENCES DistributionCenter (center_id)
)
"""

print("속성 그래프를 생성 중입니다...")
client.query(create_graph_sql).result()
print("속성 그래프(product_recommendation_graph)가 성공적으로 등록되었습니다.")

## Step 4: 특정 상품 주문 고객 네트워크 분석 및 네이티브 시각화

특정 인기 상품(예: `product_id = 25346`)을 중심으로 **[공급 물류센터 ➡️ 상품 ➡️ 구매한 여러 고객들 ➡️ 고객들의 거주 지역]**으로 확장되는 다차원 관계를 GQL로 한 번에 조회합니다.

Colab Enterprise의 네이티브 그래프 시각화 기능(`--graph display_only`)을 사용하여 1개의 상품 노드를 중심으로 여러 고객들이 별 모양(Star)으로 연결되는 시각적 네트워크를 인라인으로 확인합니다.

In [ ]:
%%bigquery --graph display_only
SELECT
  TO_JSON(c) AS Customer_Node,
  TO_JSON(p) AS Product_Node,
  TO_JSON(o) AS Ordered_Edge,
  TO_JSON(dc) AS DC_Node,
  TO_JSON(sf) AS Supplied_Edge,
  TO_JSON(l) AS Location_Node,
  TO_JSON(lives) AS Lives_In_Edge
FROM GRAPH_TABLE(
  thelook_network.product_recommendation_graph
  MATCH (dc:DistributionCenter)<-[sf:supplied_from]-(p:Product)<-[o:ordered]-(c:Customer)-[lives:lives_in]->(l:Location)
  WHERE p.product_id = 25346  -- 분석 대상 특정 상품 ID
  RETURN c, p, o, dc, sf, l, lives
)
LIMIT 100;